Read from Bronze

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import IntegerType, FloatType

spark = SparkSession.builder.getOrCreate()

bronze_path ="/Volumes/workspace/default/airlines/bronze/flights/"
bronze_df = spark.read.format("delta").load(bronze_path)
print(f" Bronze rowz loaded: {bronze_df.count():,}")


 Bronze rowz loaded: 538,837


In [0]:
#Clean data
#drop rows with missing essentials
#Fix data types
#fill null delays with 0
silver_df = (bronze_df.dropna(subset=["FlightDate","Origin","Dest","Reporting_Airline"])
            .withColumn("FlightDate", f.to_date("FlightDate", "yyyy-MM-dd"))
            .withColumn("ArrDelay", f.col("ArrDelay").cast("float"))
            .withColumn("DepDelay", f.col("DepDelay").cast("float"))
            .withColumn("Distance", f.col("Distance").cast(IntegerType()))
            .withColumn("Cancelled", f.col("Cancelled").cast(IntegerType()))
            .fillna({"ArrDelay":0.0, "DepDelay": 0.0, "CarrierDelay":0.0, "WeatherDelay":0.0, "NASDelay":0.0, "LateAircraftDelay":0.0})
            )


print(f" Silver rowz loaded after cleaning: {silver_df.count():,}")

 Silver rowz loaded after cleaning: 538,837


In [0]:
#Remove outlier delays (opver 600 mins =10 hours is unrealistic)
#Remove routes with less than 5 flights

silver_df_clean  = silver_df.filter(
    (f.col("ArrDelay")<= 600)&
    (f.col("ArrDelay")>-120) &
    (f.col("DepDelay") <= 600)
    )

print(f"** Before cleaning: {silver_df.count():,}")
print(f"** After cleaning: {silver_df_clean.count():,}")

#Overwrite Silver with clean data
silver_df_clean.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/airlines/silver/flights/")

print("** Silver layer updated with outlier removal")

** Before cleaning: 538,837
** After cleaning: 538,062
** Silver layer updated with outlier removal


In [0]:
#check the routes with very negative delays


In [0]:
#Add new columns
silver_df=silver_df_clean
silver_df= (silver_df
            #Time columns
            .withColumn("DayOfWeek", f.dayofweek("FlightDate"))
            .withColumn("Month", f.month("FlightDate"))
            .withColumn("DayName", f.date_format("FlightDate", "EEEE"))
            #Route Key
            .withColumn("Route",f.concat_ws("-",f.col("Origin"), f.col("Dest")))
            #Delay category
            .withColumn("DelayCategory", f.when(f.col("Cancelled")==1, "Cancelled")
                                          .when(f.col("ArrDelay")<= 0 , "On Time")
                                          .when(f.col("ArrDelay")<=15, "Minor (1-15m)")
                                          .when(f.col("ArrDelay")<=60, "Moderate (16-60m)")
                                          .otherwise("Severe (60+m)"))
            # Primary delay reason
            .withColumn("PrimaryDelayReason",
                        f.when(f.col("CarrierDelay")>0, "Carrier Delay")
                        .when(f.col("WeatherDelay")>0, "Weather Delay")
                        .when(f.col("NASDelay")>0, "NAS Delay")
                        .when(f.col("LateAircraftDelay")>0, "LateAircraft")
                        .otherwise("None/ On Time")
            )
)
print("New columns added!")
display(silver_df.select("FlightDate","DayOfWeek", "Month", "DayName", "Reporting_Airline", "Route", "ArrDelay", "DelayCategory", "PrimaryDelayReason").limit(10))


New columns added!


FlightDate,DayOfWeek,Month,DayName,Reporting_Airline,Route,ArrDelay,DelayCategory,PrimaryDelayReason
2023-01-24,3,1,Tuesday,DL,LAX-SEA,-6.0,On Time,None/ On Time
2023-01-25,4,1,Wednesday,DL,LAX-SEA,-5.0,On Time,None/ On Time
2023-01-26,5,1,Thursday,DL,LAX-SEA,-9.0,On Time,None/ On Time
2023-01-27,6,1,Friday,DL,LAX-SEA,41.0,Moderate (16-60m),LateAircraft
2023-01-28,7,1,Saturday,DL,LAX-SEA,-20.0,On Time,None/ On Time
2023-01-29,1,1,Sunday,DL,LAX-SEA,-16.0,On Time,None/ On Time
2023-01-30,2,1,Monday,DL,LAX-SEA,0.0,On Time,None/ On Time
2023-01-31,3,1,Tuesday,DL,LAX-SEA,-6.0,On Time,None/ On Time
2023-01-01,1,1,Sunday,DL,DTW-MCI,0.0,On Time,None/ On Time
2023-01-01,1,1,Sunday,DL,MCI-DTW,23.0,Moderate (16-60m),NAS Delay


#Save as silver delta table

In [0]:
silver_path= "/Volumes/workspace/default/airlines/silver/flights/"

silver_df.write.format("delta").mode("overwrite").save(silver_path)

spark.sql(f"""
          Create table if not exists silver_flights
          USING DELTA
          AS select* from delta.`/Volumes/workspace/default/airlines/silver/flights`
          """)

print("Silver layer saved!")
print("SILVER LAYER COMPLETE! ")



Silver layer saved!
SILVER LAYER COMPLETE! 


In [0]:
#DQ checks
display(silver_df_clean.filter((silver_df_clean.Origin == "BOS") & (silver_df_clean.Dest == "VPS")
).select("FlightDate", "Origin", "Dest", "ArrDelay", "DepDelay", "Cancelled", "Distance"))

FlightDate,Origin,Dest,ArrDelay,DepDelay,Cancelled,Distance


In [0]:
#Checking DQ- check Routes with -ve delays
display(silver_df.filter(f.col("ArrDelay")< -45).groupBy("Origin", "Dest", "Route").agg(f.count("*").alias("TotalFlights"), f.avg("ArrDelay").alias("AvgArrDelay")).orderBy("AvgArrDelay").limit(10))

Origin,Dest,Route,TotalFlights,AvgArrDelay
CMH,SEA,CMH-SEA,2,-62.5
GRR,PHX,GRR-PHX,3,-62.0
KOA,SEA,KOA-SEA,2,-61.0
FOD,ORD,FOD-ORD,1,-61.0
SYR,MCO,SYR-MCO,1,-61.0
BOS,AUS,BOS-AUS,3,-60.0
MSP,SJU,MSP-SJU,1,-60.0
LAX,KOA,LAX-KOA,2,-59.0
BUF,DEN,BUF-DEN,5,-59.0
MSP,PSP,MSP-PSP,1,-59.0
